In [ ]:
#pip install -U datasets fsspec transformers

In [2]:
from datasets import load_dataset
from transformers import BertTokenizer,BertForSequenceClassification, Trainer, TrainingArguments

In [3]:
dataset=load_dataset("imdb")

In [42]:
train_dataset=dataset["train"].select(range(1000))
print(train_dataset['label'][0])
print(train_dataset.column_names)
test_dataset=dataset["test"].select(range(500))

0
['text', 'label']


In [5]:
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\INMOR14\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
def tokenize(example):
    return tokenizer(example['text'],truncation=True,padding="max_length",max_length=256)

In [21]:
def prepocess(ds):
    ds=ds.map(tokenize,batched=True,remove_columns=['text'])
    print(ds)
    ds=ds.rename_column("label","labels")
    ds.set_format(type="torch",columns=["input_ids","attention_mask","labels"])
    return ds

In [22]:
train_dataset=prepocess(train_dataset)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})


In [23]:
test_dataset=prepocess(test_dataset)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})


In [24]:
#load model
model=BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 809.39it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those pa

In [ ]:


# for param in model.bert.parameters():
#     param.requires_grad=False

# #unfreeze last 2 layers

# for layer in model.bert.encoder.layer[-2:]:
#     for param in layer.parameters():
#         param.requires_grad=True

In [ ]:
#training arguments

training_args=TrainingArguments(
    output_dir="./bert-finetuned-imdb",
    logging_dir="./logs",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    report_to="none",
    weight_decay=0.01,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'save_safetensors'

In [27]:

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [28]:
trainer.train()

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.03it/s]


TrainOutput(global_step=125, training_loss=0.02453092956542969, metrics={'train_runtime': 508.4386, 'train_samples_per_second': 1.967, 'train_steps_per_second': 0.246, 'total_flos': 131555527680000.0, 'train_loss': 0.02453092956542969, 'epoch': 1.0})

In [29]:
%tensorboard --logdir=./logs

UsageError: Line magic function `%tensorboard` not found.


In [30]:
trainer.save_model("./bert-finetuned-imdb")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.63it/s]


In [31]:
tokenizer.save_pretrained("./bert-finetuned-imdb")

('./bert-finetuned-imdb\\tokenizer_config.json',
 './bert-finetuned-imdb\\tokenizer.json')

In [32]:
#evaluate
metrics=trainer.evaluate()

c:\Users\INMOR14\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [33]:
print(metrics)

{'eval_loss': 0.0014430989976972342, 'eval_runtime': 47.1598, 'eval_samples_per_second': 10.602, 'eval_steps_per_second': 1.336, 'epoch': 1.0}


In [36]:
#prediction

tokenizer=BertTokenizer.from_pretrained("./bert-finetuned-imdb")
model=BertForSequenceClassification.from_pretrained("./bert-finetuned-imdb")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 805.73it/s, Materializing param=classifier.weight]                                      


In [37]:
from transformers import pipeline

classifier=pipeline("text-classification",model=model,tokenizer=tokenizer)

In [44]:
text="my friend aswathy says dude is not a good movie"
result=classifier(text)

In [51]:
result[0]["label"] = "positive" if result[0]["label"] == "LABEL_0" else "negative"
print(result)
#label0-positive
#label1-negative


[{'label': 'positive', 'score': 0.996323823928833}]


In [53]:
%pip install ipywidgets

     -------------------------------------- 139.8/139.8 kB 1.7 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 4.1 MB/s eta 0:00:00
     -------------------------------------- 914.9/914.9 kB 5.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
from huggingface_hub import notebook_login
notebook_login()

In [56]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6897034d0716873fac511d57', 'name': 'moulee7788', 'fullname': 'Mouleeswaran Ranaganathan', 'email': 'moulee540@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1772323200, 'isPro': False, 'avatarUrl': '/avatars/c6039502ee80ad74100cca05d85f9c6f.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'llm_ft_write', 'role': 'write', 'createdAt': '2026-02-08T14:37:54.765Z'}}}


In [57]:
tokenizer.push_to_hub("moulee7788/my-bert-imdb")


CommitInfo(commit_url='https://huggingface.co/moulee7788/my-bert-imdb/commit/82cf5288a3cc939a044a296ae90b7f350d723cc5', commit_message='Upload tokenizer', commit_description='', oid='82cf5288a3cc939a044a296ae90b7f350d723cc5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/moulee7788/my-bert-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='moulee7788/my-bert-imdb'), pr_revision=None, pr_num=None)

In [ ]:
trainer.model.save_pretrained("bert-finetuned-imdb", safe_serialization=False)

In [63]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="bert-finetuned-imdb",
    repo_id="moulee7788/my-bert-imdb",
    repo_type="model",
)

Processing Files (6 / 7): 100%|█████████▉| 1.75GB / 1.75GB,  101kB/s  
New Data Upload: 100%|██████████| 1.44GB / 1.44GB,  101kB/s  


CommitInfo(commit_url='https://huggingface.co/moulee7788/my-bert-imdb/commit/c2c2ac7023c62d636c72838c5e7f1f2fe015ee2c', commit_message='Upload folder using huggingface_hub', commit_description='', oid='c2c2ac7023c62d636c72838c5e7f1f2fe015ee2c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/moulee7788/my-bert-imdb', endpoint='https://huggingface.co', repo_type='model', repo_id='moulee7788/my-bert-imdb'), pr_revision=None, pr_num=None)